# 06. Sorting, Reindexing & Reshaping Mechanics: Beginner Guide

### 📌 Overview & Architectural Context
Welcome to **06. Sorting, Reindexing & Reshaping Mechanics**. Transforming tabular data between wide reporting matrices and tidy long-form observations is central to financial analysis and machine learning pipelines. This notebook covers sorting by values and indices, index conforming with .reindex(), index manipulation (.set_index(), .reset_index()), pivoting (.pivot(), .pivot_table()), unpivoting (pd.melt()), multi-dimensional hierarchical stacking/unstacking (.stack(), .unstack()), and ordinal ranking mechanics.

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Sorting Values: DataFrame.sort_values()
- [x] 🔹 Sorting Index: DataFrame.sort_index()
- [x] 🔹 Conforming Labels: DataFrame.reindex()
- [x] 🔹 Resetting Index: DataFrame.reset_index()
- [x] 🔹 Setting Index: DataFrame.set_index()
- [x] 🔹 Unpivoting Wide to Long: pd.melt()
- [x] 🔹 Long to Wide Pivoting: DataFrame.pivot()
- [x] 🔹 Aggregated Pivot Tables: DataFrame.pivot_table()
- [x] 🔹 Stacking Columns to Rows: DataFrame.stack()
- [x] 🔹 Unstacking Rows to Columns: DataFrame.unstack()
- [x] 🔹 Ranking & Dense Ranking: DataFrame.rank() / Series.rank()


In [1]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import sys
import time
import os
import sqlite3
import matplotlib.pyplot as plt

# Load raw transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)
print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.head(2))

Pandas Version: 2.2.2
Loaded raw_transactions.csv: 15000 rows, 11 columns
  transaction_id customer_id merchant_id  transaction_amount card_type  \
0       TX109326      C55082       M3549              607.78      Visa   
1       TX106376      C76616       M3068             1819.11      Visa   

  transaction_status device_type  account_age_months     transaction_date  \
0           Reversed      Mobile                   8  2026-02-17 08:28:57   
1            Pending         POS                  28          03-Jan-2025   

  region  is_fraud  
0  North         0  
1   West         1  


### 🔹 Sorting Values: `DataFrame.sort_values()`
- **What it does:** Sorts DataFrame rows or columns along an axis by the values of one or more specified columns.
- **Syntax:** `DataFrame.sort_values()`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** Passing a list of booleans to `ascending` allows sorting multiple columns in mixed directions (e.g. `[True, False]`).
- **Dataset Application & Code Demonstration:** Applies Sorting Values on fintech records using columns `card_type`, `is_fraud`, `transaction_amount`, `transaction_id` to demonstrate real-world execution.


In [2]:
sorted_tx = df.sort_values(by='transaction_amount', ascending=False)
print('Top 3 Largest Transactions:\n', sorted_tx[['transaction_id', 'transaction_amount', 'card_type', 'is_fraud']].head(3))

Top 3 Largest Transactions:
       transaction_id  transaction_amount   card_type  is_fraud
5214        TX113073             1999.98  MasterCard         0
11115       TX101707             1999.85        Amex         1
7173        TX108075             1999.74  MasterCard         1


### 🔹 Sorting Index: `DataFrame.sort_index()`
- **What it does:** Sorts the DataFrame by index labels along the specified axis.
- **Syntax:** `DataFrame.sort_index()`
- **Key Note:** Sorting by index allows lightning-fast binary search slicing on large DataFrames.
- **Dataset Application & Code Demonstration:** Applies Sorting Index on fintech records using columns `transaction_date` to demonstrate real-world execution.


In [3]:
ts_df = df.set_index('transaction_date')
print('Sorted by Index Head:\n', ts_df.head(2))

Sorted by Index Head:
                     transaction_id customer_id merchant_id  \
transaction_date                                             
2026-02-17 08:28:57       TX109326      C55082       M3549   
03-Jan-2025               TX106376      C76616       M3068   

                     transaction_amount card_type transaction_status  \
transaction_date                                                       
2026-02-17 08:28:57              607.78      Visa           Reversed   
03-Jan-2025                     1819.11      Visa            Pending   

                    device_type  account_age_months region  is_fraud  
transaction_date                                                      
2026-02-17 08:28:57      Mobile                   8  North         0  
03-Jan-2025                 POS                  28   West         1  


### 🔹 Conforming Labels: `DataFrame.reindex()`
- **What it does:** Conforms DataFrame to a new index with optional filling logic, placing NA where values were absent.
- **Syntax:** `DataFrame.reindex()`
- **Key Note:** Reindexing guarantees alignment to an external reference template or time series calendar.
- **Dataset Application & Code Demonstration:** Applies Conforming Labels on fintech records using columns `transaction_amount`, `transaction_id` to demonstrate real-world execution.


In [4]:
reindexed_tx = df.head(5).reindex([0, 1, 2, 999], fill_value=0.0)
print('Reindexed Transaction Slice:\n', reindexed_tx[['transaction_id', 'transaction_amount']])

Reindexed Transaction Slice:
     transaction_id  transaction_amount
0         TX109326              607.78
1         TX106376             1819.11
2         TX103301               64.08
999            0.0                0.00


### 🔹 Resetting Index: `DataFrame.reset_index()`
- **What it does:** Resets the index of the DataFrame, moving index levels back into standard columns or replacing them with a default integer index.
- **Syntax:** `DataFrame.reset_index()`
- **Key Note:** Use `drop=True` when discarding an arbitrary or filtered index to avoid creating unwanted `'index'` columns.
- **Dataset Application & Code Demonstration:** Applies Resetting Index on fintech records using columns `transaction_amount`, `transaction_id` to demonstrate real-world execution.


In [5]:
print('Reset Index Head:\n', sorted_tx.reset_index(drop=True)[['transaction_id', 'transaction_amount']].head(3))

Reset Index Head:
   transaction_id  transaction_amount
0       TX113073             1999.98
1       TX101707             1999.85
2       TX108075             1999.74


### 🔹 Setting Index: `DataFrame.set_index()`
- **What it does:** Sets the DataFrame index (row labels) using one or more existing columns.
- **Syntax:** `DataFrame.set_index()`
- **Key Note:** Setting a high-cardinality ID or timestamp column as the index enables fast lookup speeds with `.loc[]`.
- **Dataset Application & Code Demonstration:** Applies Setting Index on fintech records using columns `transaction_id` to demonstrate real-world execution.


In [6]:
tx_indexed = df.set_index('transaction_id')
print('Transaction ID Indexed Head:\n', tx_indexed.head(2))

Transaction ID Indexed Head:
                customer_id merchant_id  transaction_amount card_type  \
transaction_id                                                         
TX109326            C55082       M3549              607.78      Visa   
TX106376            C76616       M3068             1819.11      Visa   

               transaction_status device_type  account_age_months  \
transaction_id                                                      
TX109326                 Reversed      Mobile                   8   
TX106376                  Pending         POS                  28   

                   transaction_date region  is_fraud  
transaction_id                                        
TX109326        2026-02-17 08:28:57  North         0  
TX106376                03-Jan-2025   West         1  


### 🔹 Unpivoting Wide to Long: `pd.melt()`
- **What it does:** Unpivots a DataFrame from wide format to long format, leaving identifier variables set and melting measured attributes into rows.
- **Syntax:** `pd.melt(frame, id_vars=None, value_vars=None, var_name=None, value_name='value', col_level=None, ignore_index=True)`
- **Key Note:** `pd.melt()` transforms wide tabular metrics into tidy long-form data optimal for statistical modeling and plotting libraries.
- **Dataset Application & Code Demonstration:** Unpivots wide regional quarterly revenue metrics into tidy long-form transaction records.


In [7]:
melted_tx = pd.melt(df.head(5), id_vars=['transaction_id'], value_vars=['transaction_amount', 'account_age_months'], var_name='metric', value_name='metric_value')
print('Melted Transaction Metrics:\n', melted_tx)

Melted Transaction Metrics:
   transaction_id              metric  metric_value
0       TX109326  transaction_amount        607.78
1       TX106376  transaction_amount       1819.11
2       TX103301  transaction_amount         64.08
3       TX110701  transaction_amount       1025.73
4       TX103284  transaction_amount        772.74
5       TX109326  account_age_months          8.00
6       TX106376  account_age_months         28.00
7       TX103301  account_age_months         91.00
8       TX110701  account_age_months         50.00
9       TX103284  account_age_months          5.00


### 🔹 Long to Wide Pivoting: `DataFrame.pivot()`
- **What it does:** Reshapes data from long format to wide format based on column values (requires unique index-column pairs).
- **Syntax:** `DataFrame.pivot()`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** If duplicates exist for index-column combinations, `.pivot()` raises `ValueError`; use `.pivot_table()` instead.
- **Dataset Application & Code Demonstration:** Applies Long to Wide Pivoting on fintech records using columns `card_type`, `customer_id` to demonstrate real-world execution.


In [8]:
piv = df.pivot_table(index='customer_id', columns='card_type', values='transaction_amount', aggfunc='mean')

### 🔹 Aggregated Pivot Tables: `DataFrame.pivot_table()`
- **What it does:** Creates a spreadsheet-style pivot table as a DataFrame, aggregating values across cross-classified categories.
- **Syntax:** `DataFrame.pivot_table()`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** Setting `margins=True` automatically adds total row and column marginal sums.
- **Dataset Application & Code Demonstration:** Applies Aggregated Pivot Tables on fintech records using columns `card_type`, `region`, `transaction_amount` to demonstrate real-world execution.


In [9]:
piv_spending = df.pivot_table(index='region', columns='card_type', values='transaction_amount', aggfunc='mean', margins=True)
print('Regional Card Spending Matrix (Mean Amount):\n', piv_spending.round(2))

Regional Card Spending Matrix (Mean Amount):
 card_type     Amex  Discover  MasterCard     Visa      All
region                                                    
 East      1052.77    832.92     1147.77   841.70   975.55
 North     1006.64   1169.58     1164.63   914.34  1078.73
 South     1041.77   1047.69     1126.32  1222.24  1096.62
 West       818.63   1068.40      938.36   808.63   899.14
East       1000.96   1006.02     1001.76   969.95   994.70
North       986.13   1014.09      994.79  1013.21  1002.27
South       987.27   1017.76     1008.33  1011.52  1005.98
West       1029.77   1016.82      986.61  1002.68  1009.07
east       1055.34   1153.96     1254.71   954.76  1111.78
north      1030.89    946.74     1033.94  1321.96  1100.16
south      1126.37   1037.64     1171.81  1014.28  1089.79
west       1216.81   1301.80      909.85  1054.59  1119.38
All        1002.94   1016.84     1002.26   999.25  1005.33


### 🔹 Stacking Columns to Rows: `DataFrame.stack()`
- **What it does:** Stacks prescribed level(s) from columns to index, producing a reshaped DataFrame or Series with a MultiIndex row hierarchy.
- **Syntax:** `DataFrame.stack()`
- **Key Note:** Stacking moves columns inward to rows, pivoting the table from wide to tall format.
- **Dataset Application & Code Demonstration:** Demonstrates Stacking Columns to Rows with practical fintech data structures and variables in the following code block.


In [10]:
stacked_piv = piv_spending.stack()
print('Stacked Pivot Series Head:\n', stacked_piv.head())

Stacked Pivot Series Head:
 region  card_type 
East    Amex          1052.769444
        Discover       832.917059
        MasterCard    1147.773000
        Visa           841.696667
        All            975.553973
dtype: float64


### 🔹 Unstacking Rows to Columns: `DataFrame.unstack()`
- **What it does:** Pivots a level of the (often hierarchical) index labels to the column axis, expanding the DataFrame horizontally.
- **Syntax:** `DataFrame.unstack()`
- **Key Note:** `.unstack()` is the inverse operation of `.stack()`.
- **Dataset Application & Code Demonstration:** Demonstrates Unstacking Rows to Columns with practical fintech data structures and variables in the following code block.


In [11]:
print('Unstacked DataFrame:\n', stacked_piv.unstack().head(2))

Unstacked DataFrame:
 card_type         Amex     Discover   MasterCard        Visa          All
region                                                                   
East       1052.769444   832.917059  1147.773000  841.696667   975.553973
North      1006.639231  1169.578800  1164.634118  914.338235  1078.726528


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔹 Ranking & Dense Ranking: `DataFrame.rank()` / `Series.rank()`
- **What it does:** Computes numerical data ranks (1 through N) along an axis, with customizable tie-breaking methods (`'dense'`, `'min'`, `'first'`, `'average'`, `'max'`).
- **Syntax:** `Series.rank(axis=0, method='average', numeric_only=False, na_option='keep', ascending=True, pct=False)`
  - **Parameters:**
    - `method` (*{'dense', 'min', 'first', 'average', 'max'}*, default `'average'`): How to break ties among identical values.
      - `'dense'`: Assigns consecutive ranks without gaps (e.g., 1, 2, 2, 3). Direct equivalent to SQL `DENSE_RANK()`.
      - `'min'`: Assigns lowest rank of the tie group, skipping subsequent ranks (e.g., 1, 2, 2, 4). Equivalent to SQL `RANK()`.
      - `'first'`: Assigns ranks in order of appearance (e.g., 1, 2, 3, 4). Equivalent to SQL `ROW_NUMBER()`.
      - `'average'`: Assigns average of tied ranks (e.g., 1, 2.5, 2.5, 4).
      - `'max'`: Assigns highest rank of the tie group (e.g., 1, 3, 3, 4).
    - `ascending` (*bool*, default `True`): If `False`, largest values receive Rank 1.
    - `pct` (*bool*, default `False`): If `True`, computes relative percentile rank normalized between 0.0 and 1.0.
    - `na_option` (*{'keep', 'top', 'bottom'}*, default `'keep'`): Placement behavior for missing `NaN` values.
- **Key Note:** Combined with `groupby()`, `df.groupby('col')['val'].rank(method='dense')` replicates SQL window functions: `DENSE_RANK() OVER (PARTITION BY col ORDER BY val DESC)`.
- **Dataset Application & Code Demonstration:** Demonstrates tie-breaking mechanisms, window partition ranking across regions, and top-N partition filtering on fintech transactions.


In [12]:
# Demonstration 1: Comparing tie-breaking methods (dense, min, first, average)
score_sample = pd.DataFrame({
    'customer_id': ['C101', 'C102', 'C103', 'C104', 'C105'],
    'score': [100, 90, 90, 80, 70]
})
score_sample['dense_rank'] = score_sample['score'].rank(method='dense', ascending=False).astype(int)
score_sample['sql_rank'] = score_sample['score'].rank(method='min', ascending=False).astype(int)
score_sample['row_number'] = score_sample['score'].rank(method='first', ascending=False).astype(int)
score_sample['avg_rank'] = score_sample['score'].rank(method='average', ascending=False)
print('=== Tie-Breaking Methods Comparison ===')
print(score_sample)

# Demonstration 2: Window Partition Ranking (Top 2 Transactions Per Region)
tx_clean = df.copy()
tx_clean['region'] = tx_clean['region'].astype(str).str.strip().str.title()
tx_clean['transaction_status'] = tx_clean['transaction_status'].astype(str).str.strip().str.title()
tx_clean['transaction_amount'] = pd.to_numeric(tx_clean['transaction_amount'], errors='coerce').fillna(0)

# Filter for Completed transactions and calculate dense rank partitioned by region
completed_tx = tx_clean[tx_clean['transaction_status'] == 'Completed'].copy()
completed_tx['region_rank'] = (
    completed_tx.groupby('region')['transaction_amount']
    .rank(method='dense', ascending=False)
    .astype(int)
)

top2_by_region = (
    completed_tx[completed_tx['region_rank'] <= 2]
    .sort_values(by=['region', 'region_rank'])
    [['region', 'region_rank', 'transaction_id', 'customer_id', 'transaction_amount', 'card_type']]
)
print('\n=== Top 2 Transactions Per Region (Dense Window Rank) ===')
print(top2_by_region.head(8))


=== Tie-Breaking Methods Comparison ===
  customer_id  score  dense_rank  sql_rank  row_number  avg_rank
0        C101    100           1         1           1       1.0
1        C102     90           2         2           2       2.5
2        C103     90           2         2           3       2.5
3        C104     80           3         4           4       4.0
4        C105     70           4         5           5       5.0



=== Top 2 Transactions Per Region (Dense Window Rank) ===
      region  region_rank transaction_id customer_id  transaction_amount  \
1422    East            1       TX106643      C89415             1998.49   
1708    East            2       TX108086      C51467             1996.93   
7498    East            2       TX104431      C27781             1996.93   
3099   North            1       TX113444      C13478             1998.96   
2221   North            2       TX105907      C28726             1996.02   
11115  South            1       TX101707      C30635             1999.85   
5848   South            2       TX113576      C78522             1997.32   
12873   West            1       TX110389      C69429             1998.04   

        card_type  
1422   MasterCard  
1708   MasterCard  
7498   MasterCard  
3099         Visa  
2221     Discover  
11115        Amex  
5848   MasterCard  
12873  MasterCard  


### 🔍 Scenario: Q1: Regional Fraud Risk Contingency Pivot Table
- **Objective:** Q1: Regional Fraud Risk Contingency Pivot Table
- **Approach:** Generate a pivot contingency matrix showing fraud rates across regions and device types.
- **Syntax:** `df.pivot_table(index='region', columns='device_type', values='is_fraud', aggfunc='mean') * 100`

In [13]:
fraud_matrix = df.pivot_table(index='region', columns='device_type', values='is_fraud', aggfunc='mean') * 100
print('Regional Device Fraud Rate Matrix (%):\n', fraud_matrix.round(2))

Regional Device Fraud Rate Matrix (%):
 device_type    ATM  Desktop  Mobile    POS
region                                    
 East         8.70     9.09    0.00  14.29
 North        0.00     4.76   18.75  10.53
 South       11.11    23.81   16.67   4.55
 West         6.25     0.00    0.00  13.04
East          7.67     9.75   12.18   9.83
North        11.40     9.93   10.53  13.76
South        11.70    10.11   10.90   9.42
West         14.49     9.24    8.75  12.38
east         11.54    25.00   15.00  14.29
north        13.33    18.18   20.00  27.27
south         8.33     7.41   12.50  12.50
west         10.53    23.53    4.35  11.76
